# <center><u> **COMP9444 Group Assignment - (Nom Nom Nom)** </u></center>
### <center>"Question Answering on Movie Knowledge Graphs"</center>
###### <center>Lachlan Lloyd (z5764009), Daniel Guo (z5590943), Adit Mahidadia (z5588982), Brandon Zhang (z5592981)</center>
                                                                                                    

-----------------------------------

## **Introduction**

Knowledge graphs (KGs) and natural language processing (NLP) are often regarded as quite specialised topics within machine learning, yet the systems built upon them are used extensively across many fields. Whenever a user searches the web, asks Siri or Alexa a question, receives movie recommendations on Netflix or is shown a recommended advertisement, they are, by extension, interacting with a knowledge graphs and the question answering models built on top of them. This project examines how these systems operate by developing and evaluating models that answers natural language questions using a KG. The work progresses from understanding the structure of the KG, to training models capable of traversing it, to evaluating model performance on unseen questions.

### **Problem Statement**

A knowledge graph represents facts as triples of the form (*head entity*, *relation*, *tail entity*), for example (*Chopper*, *directed_by*, *Andrew Dominik*). Knowledge Graph Question Answering (KGQA) is the task of mapping a natural language question, such as "who directed Chopper?", to a correct entity in the graph. Reliable KGQA has significant practical value in areas like recommender systems, pharmaceutical drug and hazard relationships or basic voice assistants like Alexa and Siri. For instance, if the KG contains data about a small company's services and operations, clients may be able to use a KGQA model as a basic answering tool for company-specific questions.


This project trains a model on "MetaQA", a movie based KG with over 400,000 question-answer pairs. It is capable of answering unseen test questions about cast, directors, genres and release years. Typical KGQA tasks have two main problems or limitations: **Multi-Hop Reasoning and Graph Completeness**

- Multi-hop reasoning: many questions require traversing multiple edges of the graph, including inverse relations (e.g. "which genres are the movies directed by the director of X?"), and model accuracy will typically degrades as the number of hops increases
- Graph Completeness: many real-world KGs contain missing links so the correct answer may not be reachable through a direct path. Methods that restrict possible answers to a fixed "neighbourhood" around the topic entity can end up excludeing the true answer entirely


To address these challenges, we explore existing propositions and develop implementations for them. We train and evaluate these implementations as our task demands, but also experiment with extensions on the existing solutions in an attempt to generate more accurate responses to factual questions about movies. The 3 selected baseline models we will develop, train and evaluate are the Variational Reasoning Network (VRN) [1], EmbedKGQA [2] and PullNet [3].


[1] Y. Zhang, H. Dai, Z. Kozareva, A. J. Smola, and L. Song, “Variational Reasoning for Question Answering with Knowledge Graph,” Nov. 27, 2017, arXiv: arXiv:1709.04071. doi: 10.48550/arXiv.1709.04071.

[2] A. Saxena, A. Tripathi, and P. Talukdar, “Improving Multi-hop Question Answering over Knowledge Graphs using Knowledge Base Embeddings,” in Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics, Online: Association for Computational Linguistics, 2020, pp. 4498–4507. doi: 10.18653/v1/2020.acl-main.412.

[3] H. Sun, T. Bedrax-Weiss, and W. W. Cohen, “PullNet: Open Domain Question Answering with Iterative Retrieval on Knowledge Bases and Text,” Apr. 21, 2019, arXiv: arXiv:1904.09537. doi: 10.48550/arXiv.1904.09537.

## **Data Sources**
We use the MoviE Text Audio QA, or the [MetaQA dataset](https://github.com/yuyuz/MetaQA). It contains three different components: Vanilla, NTM and Audio. Each component contains data split into three different hops, 1 hop, 2 hop and 3 hop, or the number of edges that needs to be traversed in the KG from the question to an answer. Vanilla is text data, with 1 hop data is derived from the wiki_entities branch of the WikiMovies dataset, with questions containing ambiguous entities removed. Vanilla 2 hop and 3 hop data are generated from this same knowledge base. NTM is Vanilla text that has undergone translation from English to French then back to English using a neural translation model. Audio is Vanilla text with all questions read out loud by Google's text to speech API.

Vanilla and NTM store text data as question answer pairs separated by a `\t`, with one per line. Questions are written as a knowledge triple, in the schema `subject|relation|object`. Any entities within these questions are marked by a pair of square brackets, like `question text [Entity] more question text`. Answers are written as `answer1|answer2|..|answerN` or just `answer` if only one answer is correct. Audio stores audio data as mp3 files, with one file for the question per question answer pair. Each file name follows the 0 indexed line index of the questions in Vanilla.

### **Exploratory Analysis of Data**
The question answer pairs are partitioned into test, dev and train splits. All 3 components share the same split of question answer pairs.
|       |  1 hop |   2 hop |   3 hop |
|-------|-------:|--------:|--------:|
| Train | 96,106 | 118,980 | 114,196 |
| Dev   |  9,992 |  14,872 |  14,274 |
| Test  |  9,947 |  14,872 |  14,274 |

1 hop data is made up of 13 different types of questions, 2 hop has 21 question types and 3 hop has 15 question types. Each type of question has a further 10 templates, which are different ways to word a question but have the same meanining.
- 1 hop questions:
    - Actor -> Movie
    - Director -> Movie
    - Writer -> Movie
    - Tag -> Movie
    - Movie -> Actor, Director, Writer, Genre, Language, Year, Tags, IMDb Rating, IMDb Votes
- 2 hop questions:
    - Movie -> Actor, Director, Writer -> Movie
    - Actor -> Movie -> Actor, Director, Genre, Language, Writer, Year
    - Director -> Movie -> Actor, Director, Genre, Language, Writer, Year
    - Writer -> Movie -> Actor, Director, Genre, Language, Writer, Year
- 3 hop questions:
    - Movie -> Actor -> Movie -> Director, Genre, Language, Writer, Year
    - Movie -> Director -> Movie -> Actor, Genre, Language, Writer, Year
    - Movie -> Writer -> Movie -> Actor, Director, Genre, Language, Year

The main challenge of this data is its size. Early development of models hinted that the accuracy of any model we developed was proportional to the number of samples we trained and tested against, especially for Paper 2. A lack of compute (none of our laptops are powerful enough so we relied on Kaggle or Amazon EC2 instances), we have limited our focus to just the Vanilla component of the data.

However, there is an easy way to evaluate our models using the provided data - we simply check if the model will produce an answer that's in the correct answer set of a provided question. Moreover, some models, like the Paper 2 appoach, (see below) explicitly require the main KG entity in the question when deciding potential answers. Another name for this entity is the "topic entity" of the question. The MetaQA dataset surrounded the topic entity with brackets [] for easy identification and question parsing. 

To streamline pre processing, all models we developed use a common `base.py` file, containing the classes `Example` (a question answer pair) and `KnowledgeGraph` (the KG). It also contains two helpers `parse_question` (turn a question answer pair into an `Example`) and `load_qa` (from MetaQA, load the question answer pairs for the given hop in 1, 2 or 3 hop and split in train, test or dev).

All imports and constants.

In [ ]:
from __future__ import annotations
from dataclasses import dataclass
from collections import defaultdict, deque
from collections.abc import Sequence
from pathlib import Path
from typing import Final
import re

INVERSE: Final = "inverse_"
ENTITY_REGEX: Final = re.compile(r"\[([^\]]+)\]")
WORD_REGEX: Final = re.compile(r"[a-z0-9]+")

The `Example` dataclass to represent each question answer pair, with preprocessing to extract any topic entities from the question, and extracting all words from the question string to get the template being used.

In [ ]:
@dataclass(frozen=True)
class Example:
    question: str
    topic: str
    template: str
    answers: frozenset[str]

The `parse_question` helper function to convert question answer pairs into `Example`s.

In [ ]:
def parse_question(question: str, answers: str) -> Example:
    match = ENTITY_REGEX.search(question)
    if not match:
        raise ValueError(f"Missing a [topic entity] in question: {question}")
    topic = match.group(1).strip()
    template = ENTITY_REGEX.sub(" ENTITY ", question, count=1).lower()
    template = " ".join(WORD_REGEX.findall(template))
    answer_set = frozenset(answer.strip() for answer in answers.split("|"))
    if not answer_set:
        raise ValueError(f"Question has no answers: {question}")
    return Example(question, topic, template, answer_set)

The `load_qa` helper function to convert a text file of question answer pairs, loaded from MetaQA's Vanilla component given the specific 1 or 2 or 3 hop and test or train or dev split.

In [ ]:
def load_qa(root: Path, hop: int, split: str) -> list[Example]:
    path = root / f"{hop}-hop" / "vanilla" / f"qa_{split}.txt"
    if not path.is_file():
        raise FileNotFoundError(f"QA file could not be found at path={path}")
    result: list[Example] = []
    with path.open(encoding="utf-8-sig") as file:
        for line_number, data in enumerate(file, 1):
            data = data.strip()
            if not data:
                continue
            try:
                question, answers = data.split("\t", maxsplit=1)
            except ValueError as e:
                raise ValueError(r"Expected QUESTION\tANSWERS at" + f"{path}:{line_number}") from e
            result.append(parse_question(question, answers))
    return result

The `KnowledgeGraph` class representing the KG of MetaQA, including some helper functions for models to use it.

In [ ]:
class KnowledgeGraph:
    def __init__(self, kb_path: Path) -> None:
        """
        Initialise the Knowledge Graph from the given file
        """
        self.edges: dict[str, dict[str, set[str]]] = defaultdict(lambda: defaultdict(set))
        self.triples = 0

        with kb_path.open(encoding="utf-8-sig") as file:
            for line_number, data in enumerate(file, 1):
                data = data.strip()
                if not data:
                    continue

                triple = data.split("|")
                if len(triple) != 3:
                    raise ValueError(f"Expected subject|relation|target at {kb_path}:{line_number}")

                subject, relation, target = (part.strip() for part in triple)

                self.edges[subject][relation].add(target)
                self.edges[target][INVERSE + relation].add(subject)
                self.triples += 1

    def follow(self, topic_entity: str, relation_path: tuple[str, ...]) -> set[str]:
        """
        Gets all final entities reachable by following the given relation path starting from the topic entity
        """
        nodes: set[str] = {topic_entity}
        for relation in relation_path:
            next_nodes = set[str]()
            for node in nodes:
                targets = self.edges.get(node, {}).get(relation, ())
                for target in targets:
                    next_nodes.add(target)
            nodes = next_nodes
            if not nodes:
                break
        return nodes

    def all_paths(self, topic_entity: str, hops: int) -> dict[tuple[str, ...], set[str]]:
        """
        Gets all combinations of relations and final entities found by following them, within hops distance from topic entity
        """
        states: dict[tuple[str, ...], set[str]] = {(): {topic_entity}}
        for _ in range(hops):
            next_states: dict[tuple[str, ...], set[str]] = defaultdict(set)
            for relation_path, nodes in states.items():
                for node in nodes:
                    for relation, targets in self.edges.get(node, {}).items():
                        next_states[relation_path + (relation,)].update(targets)
            states = next_states
            if not states:
                break
        return states

    def get_all_entities(self) -> set[str]:
        """
        Get all entities in the Knowledge Graph
        """
        return set(self.edges.keys())

    def get_all_relations(self, include_inverse: bool = True) -> set[str]:
        """
        Get all relations in the Knowledge Graph
        """
        relations = {
            relation
            for node_relations in self.edges.values()
            for relation in node_relations
        }

        if not include_inverse:
            relations = {
                relation for relation in relations
                if not relation.startswith(INVERSE)
            }

        return relations

    def get_all_triples(self, include_inverse: bool = True) -> set[tuple[str,str,str]]:
        """
        Get all facts, or (subject, relation, object) triples in the Knowledge Graph
        """
        return {
        (source, relation, target)
        for source, relations in self.edges.items()
        for relation, targets in relations.items()
        if include_inverse or not relation.startswith(INVERSE)
        for target in targets
        }

    def has_triple(self, triple: tuple[str,str,str]) -> bool:
        """
        Check if the given fact, or (subject, relation, object) triple, is in the Knowledge Graph
        """
        source, relation, target = triple
        return target in self.edges.get(source, {}).get(relation, set())

## **Models and/or Methods**
We work on 3 baseline models as provided in our task description and focus on potential improvements to each. As mentioned in the introduction, they are: Variational Reasoning Network (VRN) [1], EmbedKGQA [2] and PullNet [3].


[1] Y. Zhang, H. Dai, Z. Kozareva, A. J. Smola, and L. Song, “Variational Reasoning for Question Answering with Knowledge Graph,” Nov. 27, 2017, arXiv: arXiv:1709.04071. doi: 10.48550/arXiv.1709.04071.

[2] A. Saxena, A. Tripathi, and P. Talukdar, “Improving Multi-hop Question Answering over Knowledge Graphs using Knowledge Base Embeddings,” in Proceedings of the 58th Annual Meeting of the Association for Computational Linguistics, Online: Association for Computational Linguistics, 2020, pp. 4498–4507. doi: 10.18653/v1/2020.acl-main.412.

[3] H. Sun, T. Bedrax-Weiss, and W. W. Cohen, “PullNet: Open Domain Question Answering with Iterative Retrieval on Knowledge Bases and Text,” Apr. 21, 2019, arXiv: arXiv:1904.09537. doi: 10.48550/arXiv.1904.09537.


Since no implementation of either 3 baseline models are available, the first part of our work focuses on understanding, experimenting and developing the implementation for each model.

### **Baselines**
All imports used for the models

In [ ]:
import torch
from torch import Tensor, nn
from torch.nn import functional as F

#### **VRN (Variational Reasoning Network)**
We implement the VRN as described in the paper. We first initialise the VRN class with the knowledge graph, training examples, and specify the number of dimensions for any internal embeddings. We treat relations as bidirectional by removing their inverses from the knowledge graph. We also precompute the encodings for every possible relation and word in the knowledge graph. And finally, we initialise a projection matrix and question embeddings (see later for explanations).

In [ ]:
class VRN(nn.Module):
    def __init__(self, graph: KnowledgeGraph, train_examples: Sequence[Example], hidden_dim: int = 128) -> None:
        super().__init__()
        self.graph: KnowledgeGraph = graph

        relations = sorted({
            relation.removeprefix(INVERSE)
            for adjacency in graph.edges.values()
            for relation in adjacency
        })
        self.relation_ids: dict[str, int] = {relation: i for i, relation in enumerate(relations)}

        vocabulary = sorted({
            word
            for example in train_examples
            for word in example.template.split()
        })
        self.word_ids: dict[str, int] = {word: i for i, word in enumerate(vocabulary, 1)}

        self.projection = nn.Linear(hidden_dim + len(relations), hidden_dim, bias=False)
        self.question_embeddings = nn.Embedding(len(vocabulary) + 1, hidden_dim)

From an implementation perspective, the VRN solves QA by first identifying all reachable entities (which we will refer to as candidate answers) within the given number of hops. It will then construct one subgraph per candidate answer, containing all edges (relations) and vertices (entities) on any path from the topic entity to the candidate answer. Each subgraph is then projected to a graph embedding, which is then dot producted with the question embedding to get the logits for each candidate answer, and finally softmaxed to the candidate answer's probability.

In [ ]:
def forward(self, example: Example, hops: int) -> tuple[list[str], Tensor]:
    if hops < 0:
        raise ValueError("hops must be non-negative")
    device = self.projection.weight.device

    # Form the word matrix of the question, then use the question_embedding to embed the question
    token_ids = torch.tensor(
        [self.word_ids.get(w, 0) for w in example.template.split()],
        dtype=torch.long,
        device=device,
    )
    question: Tensor = self.question_embeddings(token_ids).sum(dim=0)

    # g(G_y->y) = 0
    embeddings: dict[str, Tensor] = {
        example.topic: self.projection.weight.new_zeros(
            self.projection.out_features
        )
    }
    layer: list[str] = [example.topic]

    # Equation 6
    for _ in range(hops):
        incoming: dict[str, list[Tensor]] = {}

        # Iterate over all entities in current layer
        for parent in layer:
            # Find all entities connected to this entity in the next layer (1 hop over)
            for relation, targets in self.graph.edges.get(parent, {}).items():
                unseen = [child for child in targets if child not in embeddings]
                if not unseen:
                    continue

                # One hot encode the relation and parent subgraph embedding, then project to current subgraph embedding
                relation_id = self.relation_ids[relation.removeprefix(INVERSE)]
                one_hot = F.one_hot(
                    torch.tensor(relation_id, device=device),
                    num_classes=len(self.relation_ids),
                ).to(self.projection.weight.dtype)
                message = F.relu(
                    self.projection(torch.cat((embeddings[parent], one_hot)))
                )

                for child in unseen:
                    incoming.setdefault(child, []).append(message)

        if not incoming:
            break

        # Average all subgraph embeddings for each entity in the next layer
        layer = sorted(incoming)
        for entity in layer:
            embeddings[entity] = torch.stack(incoming[entity]).mean(dim=0)

    # Equation 5
    candidates = list(embeddings)
    # Dot product to get logits for softmax
    logits = torch.stack(list(embeddings.values())) @ question
    return candidates, logits

We then use log loss (as our problem is classification) as the loss for VRN. As forward gives the softmax logits, we use `torch.nn.F`'s `log_softmax` function.

#### **Paper 2 Model - EmbedKGQA**

Paper 2 proposes a model called EmbedKGQA that exceeds state-of-the-art baselines (at the time). It differs from other methods to address KG sparsity and questions that require multi-hop inference, such as those that rely on augmented (denser) KGs using supporting text documents, sourced from the text corpus. This is an unreliable approach as these external documents may not always be readily available. Existing methods also unnecessarily impose pre-specified heuristic neighbourhood size limitations from which the true answer needs to be selected. But if the correct answer to a question is, say, 4 hops (edges) from the top entity (the subject of the question), and the pre-set neighbourhood limitation is 3 hops, then the right answer is impossible for the model to find. The paper’s solution, EmbedKGQA, avoids both augmented KGs and neighbourhood constraints. The model consists of three modules. 

##### **KG Embedding Module**

The first module is the KG Embedding Module (GEM), which uses ComplEx embeddings to learn a d-dimensional complex vector for all entities (node) and relations (edge) in the KG. The embeddings are learned to reduce a scoring metric for all false triplets, and increase the scoring metric above zero for all true triplets. A triplet {starting entity, edge relation, end entity} is true iff the natural language sentence derived from the triplet is true. Note that the scoring metric/function takes in three d-dimensional vectors in an order-variant manner and outputs a real number. For KG embeddings, the vectors are the embeddings of the starting entity, edge relation and the conjugate of the end entity embedding, in that order.

In [ ]:
import random
from collections.abc import Iterable

class kg_embedding_module(nn.Module):
    """ComplEx knowledge graph (KG) embedding model for triple scoring.

    Learns complex-valued embeddings for entities and relations
    and computes a compatibility ComplEx score for each (head, relation,
    tail) KG triple.

    The model learns embeddings for entities and relations in the KG such that the raw ComplEx
    score for a KG triple that exists in the KG is more than zero, and the score for a non-existent
    KG triple is less than zero. Equivalently, when a sigmoid transformation is applied to the raw
    ComplEx score, true KG triples should have sigmoid scores close to one and false triples should
    have sigmoid scores close to zero.
    
    The trainable model parameters are the complex embeddings in self.entity_embeddings and
    self.relation_embeddings. The model's input is a string triple (head entity, relation, tail entity) which is
    abbreviated as (h, r, t), and the lookups for the embeddings of each entity/relation use a string to index mapping 
    stored in this class' attributes. The target output is one for true KG triples, and zero for false triples.
    
    The calculated raw ComplEx score for a triple (h, r, t) is found with the self.forward(h, r, t)
    method, and this score undergoes a sigmoid transformation (as described in the previous paragraph).
    The loss function between the transformed score and the target output is binary cross entropy (BCE)
    loss. The BCEWithLogitsLoss class handles the sigmoid and BCE loss at once. It also computes loss for
    batch training.
    
    """

    def __init__(
        self,
        kg: KnowledgeGraph,
        embedding_dim: int = 100
    ) -> None:
        """Initialise the KG ComplEx embedding model.

        Args:
            kg (Knowledge Graph): the knowledge graph for the embeddings
            embedding_dim (int): Number of complex dimensions per embedding.
                Each dimension has a real and imaginary component, so each
                embedding stores 2 * embedding_dim real values.

                For example, embedding_dim=3 represents the complex vector
                [1+23j, -7-4j, 0-6j] as [1, -7, 0, 23, -4, -6],
                where the first half stores real components and the second
                half stores imaginary components.

        Returns:
            None: Initialises entity and relation embedding layers and their string-id mappings.

        """

        super().__init__()

        self.kg = kg
        self.embedding_dim = embedding_dim

        # Map entities and relations to unique integer IDs; we sort the sets for consistency in
        # ids across different created objects of this class, provided the KG is the same
        self.entity_to_idx = {
            entity: i for i, entity in enumerate(sorted(kg.get_all_entities()))
        }
        self.relation_to_idx = {
            relation: i for i, relation in enumerate(sorted(kg.get_all_relations()))
        }

        # We use sparse embeddings for KG embeddings, as in any given batch, only at most a few rows of the embedding matrix are updated. Setting
        # sparse=True thus speeds up the Adam optimiser, gradient calculations and embedding updates during training. 
        self.entity_embeddings = nn.Embedding(
            len(self.entity_to_idx), 2 * embedding_dim, sparse=True
        )
        self.relation_embeddings = nn.Embedding(
            len(self.relation_to_idx), 2 * embedding_dim, sparse=True
        )

        nn.init.xavier_normal_(self.entity_embeddings.weight)
        nn.init.xavier_normal_(self.relation_embeddings.weight)


    def forward(
        self,
        batch: Iterable[tuple[str, str, str]]
    ) -> torch.Tensor:
        """Compute raw ComplEx scores for a batch of string KG triples.

        If there are 1000 entities and 10 relations, and we want to find the ComplEx score of the string
        KG triple (h, r, t), then we call model.forward([(h, r, t)]). But we need a way to find the embeddings
        of the two entities h and t, and the relation r. This requires entities and relations to be
        mapped to a unique entity and relation index respectively, where this index is used to lookup the
        embedding - this is mapping is created when initialising the class, and the lookup is conducted in this
        method.

        Args:
            batch (Iterable[tuple[str, str, str]]): A collection of
            (head, relation, tail) KG triples represented as strings.
            Accepts a list, tuple, or set of KG triples. The size of this iterable
            is batch_size.        

        Returns:
            torch.Tensor: Raw compatibility scores with shape
                (batch_size,). Higher scores indicate greater
                compatibility between the head, relation and tail.
                Scores are unbounded real values.

       
        """
        batch = list(batch)

        if not batch:
            raise ValueError("Batch cannot be empty")

        device = self.entity_embeddings.weight.device

        # Convert the list of string KG triples to a list of integer triples, which represent indices
        indexed_batch = [
            (
                self.entity_to_idx[h],
                self.relation_to_idx[r],
                self.entity_to_idx[t]
            )
            for h, r, t in batch
        ]

        indices = torch.tensor(
            indexed_batch, dtype=torch.long, device=device
        )

        # Retrieve embeddings
        head = self.entity_embeddings(indices[:, 0])
        relation = self.relation_embeddings(indices[:, 1])
        tail = self.entity_embeddings(indices[:, 2])

        h_re, h_im = torch.chunk(head, 2, dim=-1)
        r_re, r_im = torch.chunk(relation, 2, dim=-1)
        t_re, t_im = torch.chunk(tail, 2, dim=-1)

        re_score = h_re * r_re - h_im * r_im
        im_score = h_re * r_im + h_im * r_re

        # In the ComplEx score formula, we use the conjugate of the tail entity's embedding, hence
        # the "+ im_score * t_im" below, not "- im_score * t_im"
        score = (
            re_score * t_re + im_score * t_im
        ).sum(dim=-1)

        return score


    def predict_proba(
        self,
        batch: Iterable[tuple[str, str, str]]
    ) -> torch.Tensor:
        """Compute sigmoid-transformed scores for string KG triples.

        Args:
            batch (Iterable[tuple[str, str, str]]): A collection of
                (head entity, relation, tail entity) KG triples represented as strings.
                Accepts a list, tuple, or set of KG triples. The size of this iterable
                is batch_size.

        Returns:
            torch.Tensor: Sigmoid-transformed scores between 0 and 1,
                with shape (batch_size,). Higher values indicate
                greater predicted plausibility. These values are
                not necessarily calibrated probabilities.
        """
        return torch.sigmoid(self.forward(batch))
    

    def fit(
        self,
        epochs: int = 100,
        batch_size: int = 128,
        learning_rate: float = 0.001,
        negatives_per_positive: int = 3
    ) -> list[float]:
        """Train ComplEx embeddings using KG triples.

        Args:
            epochs (int): Number of training epochs.
            batch_size (int): Number of positive/correct KG triples per batch.
            learning_rate (float): Adam optimizer learning rate.
            negatives_per_positive (int): Number of negative/incorrect KG triples
                sampled for each positive triple.

        Returns:
            list[float]: Average training loss for each epoch.
        """
        print("Beginning KG embedding module fitting")
        
        if epochs < 1 or batch_size < 1 or negatives_per_positive < 1:
            raise ValueError("Training parameters must be positive")

        entities = list(self.kg.get_all_entities())
        positive_triples = list(self.kg.get_all_triples())

        if not positive_triples:
            raise ValueError("No trainable KG triples found")

        device = self.entity_embeddings.weight.device
        losses = []

        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.SparseAdam(self.parameters(), lr=learning_rate)

        for epoch in range(epochs):
            print(f"Beginning epoch {epoch + 1}")
            self.train()
            random.shuffle(positive_triples)

            total_loss = 0.0
            total_samples = 0

            for start in range(0, len(positive_triples), batch_size):
                batch_positive = positive_triples[start:start + batch_size]

                # Generate negative/incorrect KG triples by corrupting tails, while keeping head and relation the
                # same
                batch_negative = []
                for h, r, _ in batch_positive:
                    for _ in range(negatives_per_positive):
                        new_t = random.choice(entities)
                    
                        # Avoid sampling an existing positive/true triple
                        attempt = 0
                        while self.kg.has_triple((h, r, new_t)):
                            if attempt > 100:
                                raise ValueError("Could not generate false KG triple!")
                            new_t = random.choice(entities)
                            attempt += 1
                    
                        batch_negative.append((h, r, new_t))

                # Combine positive and negative KG triples, and do forward pass
                batch = batch_positive + batch_negative
                scores = self.forward(batch)

                # Calculate loss           
                labels = torch.tensor(
                    [1.0] * len(batch_positive)
                    + [0.0] * len(batch_negative),
                    dtype=torch.float32,
                    device=device
                )
                loss = criterion(scores, labels)

                # Backpropagation
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                total_loss += loss.item() * len(batch)
                total_samples += len(batch)

            avg_loss = total_loss / total_samples
            losses.append(avg_loss)

            print(
                f"Epoch {epoch + 1}/{epochs}, "
                f"Loss: {avg_loss:.4f}"
            )

        self.eval()
        return losses

    def get_all_entities(self) -> set[str]:
        """
        Returns:
            set[str]: All entities in the knowledge graph supplied to this KG embedding class
        """
        return self.kg.get_all_entities()

    def get_all_relations(self) -> set[str]:
        """
        Returns:
            set[str]: All relations in the knowledge graph supplied to this KG embedding class
        """
        return self.kg.get_all_relations()


##### **Question Embedding Module**

Questions are embedded using RoBERTa (Liu et al., 2019) into a 768-dimensional vector. This is then passed through 4 fully connected linear layers with ReLU activation and finally projected onto the d-dimensional complex space, matching the vector space for the KG embedding. But how are these embeddings learnt? Consider a new triple {topic entity, question, answer entity}. The topic entity is the entity present in the question, and the answer entity is a possible entity that can be the response to the question (implying relations cannot be used as answers). The model finds an embedding strategy such that the embedding representation of the triple, when supplied to the same scoring metric used in the previous paragraph for learning KG embeddings, outputs a number above 0 for correct answers, and below 0 for incorrect answers. The model is learned by minimising the binary cross entropy loss between the sigmoid of the scores and the target labels for all possible answers (nodes/entities), where the target label is 1 for the correct answers and 0 otherwise. 

In the original paper, the conjugate of the answer entity's embedding is not taken, before passing the question triple into the scoring function. However, our group thought this was peculiar, as taking the conjugate introduces directionality into a triple, which matches the directionality of the natural language formulation of a question triple. For instance, the true triple (topic entity = "Oppenheimer", question = "Who directed Oppenheimer?", answer entity = "Christopher Nolan") is clearly different to the false question triple (topic entity = "Christopher Nolan", question = "Who directed Oppenheimer?", answer entity = "Oppenheimer"). So, we took the conjugate of the answer entity's embedding in a question triple when computing the scoring function, in both fitting/training and forward-passing for testing, similarly to the KG embedding model.

In [ ]:
import torch
import torch.nn as nn
import random
import copy
from collections.abc import Iterable
from transformers import RobertaTokenizer, RobertaModel

class question_embedding_module(nn.Module):
    """Question embeddings learnt using the complEx scoring function. 

    Learns complex-valued embeddings for questions
    and computes a compatibility ComplEx score for each (topic, question,
    answer) triple.
    """

    def __init__(
        self,
        kg_embedding_module : kg_embedding_module,
        question_training_data: list[Example],
        question_to_idx: dict[str, int] | None = None,
        roberta_question_embeddings:  torch.Tensor | None = None,
        first_hidden_layer_dim: int = 600,
        second_hidden_layer_dim: int = 450,
        third_hidden_layer_dim: int = 300,
        dropout_rate: float = 0
    ) -> None:
        """Initialise the question embedding model.

        This module embeds a natural language question q to a fixed dimension vector, where the 
        dimensionality is the same as the embeddings of the entities and relations supplied in the
        kg_embedding_module. This model constructs embeddings using a feed-forward neural network that
        first embeds the question q using RoBERTa (Liu et al., 2019) into a 768-dimensional vector. 
        This is then passed through 3 fully connected layers, excluding the input layer, with ReLU
        activation and finally projected onto the complex space C. The model's input is a question triple
        (topic entity, question, answer entity) abbreviated as (h, q, a), its output in self.forward() is a raw ComplEx
        score measuring the likelihood that the question triple is correct (that is, the answer is correct
        for the question). The learnable model parameters are the weights in the linear layers which determine
        question embedding in complex space - the embeddings of the entitites are fixed and provided
        through the kg_complex embedding object parameter.

        Unlike the kg_complex class and its static, provided entities and relations, questions are not static and we
        must be able to embed any question passed into the forward method. The question_embedding_module
        learns the process to produce an embedding for any question, including those it hasn't encountered
        before. However, because the RoBERTa model is fixed, we temporarily store question embeddings for questions
        in self.question_training_data when fitting the model.

        Args:
            kg_embedding_module (kg_embedding_module class): the trained embedding model for entities and relations,
                containing the learned embeddings for all entities and relations in the knowledge graph
            question_training_data (list[Example]): a list of questions, their topic entity, the template and
                possible answers. This tells the class what the right answers are to a certain question, which
                determines the target output for a given question triple.
            question_to_idx (dict[str, int] | None): a dictionary mapping all questions in the question_training_data
                to an integer between 0 and (self.roberta_question_embeddings.shape[0] - 1). This maps a question string
                to the index of its 768-dimensional RoBERTa embedding in the self.roberta_question_embeddings tensor.
            roberta_question_embeddings (torch.Tensor | None): a tensor of size (len(question_training_data) or greater, 768) containing
                the RoBERTa question embeddings for all questions in the training data. This is useful if the question
                embeddings are pre-computed for the entire batch, to avoid running the fixed/frozen RoBERTa model again.
                The number of rows in this tensor can be less than len(question_training_data) if there are duplicate questions
                across multiple examples. If not provided, this class will calculate the embeddings and generate its own
                question_to_idx dictionary.
            first_hidden_layer_dim (int): number of hidden nodes in the second layer (first hidden layer)
            second_hidden_layer_dim (int): number of hidden nodes in the third layer (second hidden layer)
            third_hidden_layer_dim (int): number of hidden nodes in the fourth layer (third hidden layer)
            dropout_rate (int): Randomly switch off nodes with probability dropout_rate
        Returns:
            None
        
        """
        super().__init__()

        self.kg_embedding_module = kg_embedding_module
        self.question_training_data = question_training_data
        self.question_to_idx = question_to_idx
        self.roberta_question_embeddings = roberta_question_embeddings

        # We use double the embedding dimensionality of the knowledge graph entities/relations since it is a 
        # complex vector embedding
        self.linear_relu_stack = nn.Sequential(
            nn.Linear(768, first_hidden_layer_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(first_hidden_layer_dim, second_hidden_layer_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(second_hidden_layer_dim, third_hidden_layer_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),
            nn.Linear(third_hidden_layer_dim, 2 * self.kg_embedding_module.embedding_dim),
        )      

        # We freeze the parameters in the KG embedding module
        for param in self.kg_embedding_module.parameters():
            param.requires_grad = False


    def forward(
        self,
        batch: Iterable[tuple[str, str, str]],
        question_to_idx: dict[str, int] | None = None,
        roberta_question_embeddings: torch.Tensor | None = None
    ) -> torch.Tensor:
        """Compute raw ComplEx scores for a batch of question triples.

        If we want to find the ComplEx score of the question triple (h, q, a),
        then we call model.forward([(h, q, a)]). Here, the batch_size is one.
        
        Args:
            batch (Iterable[tuple[str, str, str]]): A collection of
                (topic entity, question, answer entity) triples represented as strings.
                Accepts a list, tuple, or set of triples. The size of this iterable
                is batch_size.
            question_to_idx (dict[str, int] | None): a dictionary mapping all questions in the batch
                to an integer between 0 and (roberta_question_embeddings.shape[0] - 1). This maps a question string
                to its 768-dimensional RoBERTa embedding in the roberta_question_embeddings tensor.
            roberta_question_embeddings (torch.Tensor | None): A tensor of size (batch_size or greater, 768)
                that contains the RoBERTa question embeddings for at least each question in the batch.
                This is useful if the question embeddings are pre-computed for the entire batch,
                to avoid running the fixed/frozen RoBERTa model again.
            
            If either of the latter two parameters are None, this method will attempt to source them from the
            class' parameters. If at least one of them is still None, or if there is a question in the batch that
            isn't a key in the question_to_idx dictionary, then the method will compute the RoBERTa question embeddings
            and question_to_idx dictionary on its own, by calling the static method at the bottom of this cell.
        
        Returns:
            torch.Tensor: Raw compatibility scores with shape
                (batch_size,). Higher scores indicate greater
                compatibility between the topic entity, question and answer entity.
                Scores are unbounded real values.
        
        """
        device = next(self.parameters()).device
        
        batch = list(batch)
        if not batch:
            raise ValueError("Batch cannot be empty")

        topic_entities, questions, answer_entities = map(list, zip(*batch))

        if question_to_idx is None and roberta_question_embeddings is None:
            question_to_idx = self.question_to_idx
            roberta_question_embeddings = self.roberta_question_embeddings
        if (question_to_idx is None
            or roberta_question_embeddings is None
            or any(question not in question_to_idx for question in questions)
            ):
            print("Code should never go here in qem")
            roberta_question_embeddings, question_to_idx = (
                self.convert_questions_roberta(questions, device))
        
        # Look up the row indices for all questions in the batch
        assert roberta_question_embeddings is not None
        assert question_to_idx is not None
        question_indices = [question_to_idx[question] for question in questions]
        # Retrieve the corresponding precomputed RoBERTa embeddings
        roberta_question_embeddings = roberta_question_embeddings[question_indices].to(device)

        # The 768-dimensional embeddings for the questions are passed through three fully connected hidden layers
        # with ReLu activation, eventually reaching a complex embedding with dimensionality equal to the embeddings
        # in the knowledge graph
        question_embeddings = self.linear_relu_stack(roberta_question_embeddings)

        # Now we get the embeddings of the topic and answer entities
        topic_entity_indexed_batch = [
            self.kg_embedding_module.entity_to_idx[topic_entity]
            for topic_entity in topic_entities
        ]    
        topic_entity_indices = torch.tensor(
            topic_entity_indexed_batch, dtype=torch.long, device=device
        )
        answer_entity_indexed_batch = [
            self.kg_embedding_module.entity_to_idx[answer_entity]
            for answer_entity in answer_entities
        ]           
        answer_entity_indices = torch.tensor(
            answer_entity_indexed_batch, dtype=torch.long, device=device
        )
        head = self.kg_embedding_module.entity_embeddings(topic_entity_indices[:])
        tail = self.kg_embedding_module.entity_embeddings(answer_entity_indices[:])

        # Lastly, we compute the ComplEx scoring function
        h_re, h_im = torch.chunk(head, 2, dim=-1)
        r_re, r_im = torch.chunk(question_embeddings, 2, dim=-1)
        t_re, t_im = torch.chunk(tail, 2, dim=-1)
        
        re_score = h_re * r_re - h_im * r_im
        im_score = h_re * r_im + h_im * r_re
        
        # In the ComplEx score formula, we use the conjugate of the tail entity's embedding, hence
        # the "+ im_score * t_im" below, not "- im_score * t_im"
        score = (
            re_score * t_re + im_score * t_im
        ).sum(dim=-1)
        
        return score

    def predict_proba(
            self,
            batch: Iterable[tuple[str, str, str]],
            question_to_idx: dict[str, int] | None = None,
            roberta_question_embeddings: torch.Tensor | None = None
        ) -> torch.Tensor:
            """Compute sigmoid-transformed scores for a batch of question triples.
            
            Args:
                batch (Iterable[tuple[str, str, str]]): A collection of
                    (topic entity, question, answer entity) triples represented as strings.
                    Accepts a list, tuple, or set of triples. The size of this iterable
                    is batch_size.
                question_to_idx (dict[str, int] | None): a dictionary mapping all questions in the batch
                    to an integer between 0 and (roberta_question_embeddings.shape[0] - 1). This maps a question string
                    to its 768-dimensional RoBERTa embedding in the roberta_question_embeddings tensor.
                roberta_question_embeddings (torch.Tensor | None): A tensor of size (batch_size or greater, 768)
                    that contains the RoBERTa question embeddings for at least each question in the batch.
                    This is useful if the question embeddings are pre-computed for the entire batch,
                    to avoid running the fixed/frozen RoBERTa model again.
                            
                If either of the latter two parameters are None, the forward method will attempt to source them from the
                class' parameters. If at least one of them is still None, or if there is a question in the batch that
                isn't a key in the question_to_idx dictionary, then the forward method will compute the RoBERTa question embeddings
                and question_to_idx dictionary on its own.
            
            Returns:
                torch.Tensor: Sigmoid-transformed scores between 0 and 1,
                    with shape (batch_size,). Higher values indicate
                    greater predicted plausibility. These values are
                    not necessarily calibrated probabilities.
            """
            return torch.sigmoid(self.forward(batch, question_to_idx, roberta_question_embeddings))

    def fit(
            self,
            epochs: int = 100,
            batch_size: int = 128,
            learning_rate: float = 0.001,
            weight_decay: float = 0,
            negatives_per_positive: int = 3
        ) -> list[float]:
        """Train question embeddings using the provided list of question examples when initialising this class.
    
        Args:
            epochs (int): Number of training epochs
            batch_size (int): Number of distinct questions per batch
            learning_rate (float): Adam optimizer learning rate
            weight_decay (float): L2 regularization coefficient applied by the Adam optimizer
            negatives_per_positive (int): Number of negative/incorrect question triples
                            sampled for each positive triple.
        Returns:
            list[float]: Average training loss for each epoch.
        """
        print("Beginning question embedding module fitting")
        
        if epochs < 1 or batch_size < 1:
            raise ValueError("Training parameters must be positive")

        question_training_data = copy.deepcopy(self.question_training_data)
        entities = list(self.kg_embedding_module.get_all_entities())
        device = next(self.parameters()).device
        losses = []
        
        criterion = nn.BCEWithLogitsLoss()
        optimizer = torch.optim.Adam(self.parameters(), lr=learning_rate, weight_decay=weight_decay)

        for epoch in range(epochs):
            print(f"Beginning epoch {epoch + 1}")
            self.train()
            random.shuffle(question_training_data)
        
            total_loss = 0.0
            total_samples = 0
        
            for start in range(0, len(question_training_data), batch_size):
                # Each example in the question training data can have multiple answers, and we treat each question
                # (topic-question-answer) triple its own sample. Paper 2 does not require a template in the question
                # embedding module, so we leave out this field when constructing a batch.
                batch_examples = question_training_data[start:start + batch_size]
                batch_positive = [
                    (example.topic, example.question, answer)
                    for example in batch_examples
                    for answer in example.answers
                ]
        
                # Generate negative/incorrect question triples by corrupting the answer entity, while keeping head (topic)
                # entity and question the same
                batch_negative = []
                for example in batch_examples:
                    num_negatives = len(example.answers) * negatives_per_positive

                    for _ in range(num_negatives):
                        attempt = 0
                        while True:
                            if attempt > 100:
                                raise ValueError("Could not generate false question triple!")
                            answer = random.choice(entities)
                            attempt += 1
                            if answer not in example.answers:
                                break

                        batch_negative.append(
                            (example.topic, example.question, answer)
                        )
                
                # Combine positive and negative question triples, and do forward pass
                batch = batch_positive + batch_negative

                scores = self.forward(
                    batch,
                    question_to_idx=self.question_to_idx,
                    roberta_question_embeddings=self.roberta_question_embeddings)
        
                # Calculate loss           
                labels = torch.tensor(
                    [1.0] * len(batch_positive)
                    + [0.0] * len(batch_negative),
                    dtype=torch.float32,
                    device=scores.device
                )
                loss = criterion(scores, labels)
        
                # Backpropagation
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
        
                total_loss += loss.item() * len(batch)
                total_samples += len(batch)
        
            avg_loss = total_loss / total_samples
            losses.append(avg_loss)
        

            print(
                f"Epoch {epoch + 1}/{epochs}, "
                f"Loss: {avg_loss:.4f}"
            )
        
        self.eval()
        return losses

    @staticmethod
    def convert_questions_roberta( 
            questions: list[str],
            device: torch.device,
            batch_size: int = 32,
            verbose: bool = False
        ) -> tuple[torch.Tensor, dict[str, int]]:
        """Convert a batch of questions into 768-dimensional RoBERTa embeddings.

        Uses a frozen RoBERTa model with mean pooling over non-padding tokens, unlike the paper's
        implementation which extracts the embedding representation of the first token only.

        Args:
            questions (list[str]): List of natural language questions of length batch_size.
            device (torch.device): Device on which to perform computation.
            batch_size (int): the number of questions that have their RoBERTa embedding computing
                in each call to the frozen RoBERTa model
            verbose (bool): If true, calling this function will print updates after each batch

        Returns:
            tuple[torch.Tensor, dict[str, int]]: Embeddings of shape
                (batch_size, 768) and a dictionary mapping each question
                to its corresponding tensor row index.
        """
        roberta_model = RobertaModel.from_pretrained('roberta-base').to(device)
        roberta_model.eval()
        tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
        roberta_question_embeddings = torch.empty(
            (len(questions), 768),
            dtype=torch.float32
        )
        question_to_idx = {
            question: idx
            for idx, question in enumerate(questions)
        }
        total_batches = (len(questions) + batch_size - 1) // batch_size

        for start in range(0, len(questions), batch_size):
            batch_questions = questions[start:start + batch_size]
            # First, we tokenize the questions into tokens, then extract the numeric tokenID that
            # corresponds to each token in the vocabulary. Any padded tokens should not be considered
            # by the RobertaModel, so we use an attention_mask which has entries that are either 0 or 1. 
            # input_ids and attention_mask are a (batch_size, 64) dimension tensors. We pad every question
            # to 64 tokens to ensure consistency.
            questions_tokenized = tokenizer(
                batch_questions,
                padding=True,
                truncation=True,
                max_length=64,
                return_tensors="pt"
            )  
            input_ids = questions_tokenized["input_ids"].to(device)
            attention_mask = questions_tokenized["attention_mask"].to(device)
            
            # Then, we pass the input_ids of the tokenized questions into the RoBERTa model, along with the
            # attention_mask. The token sequence length (second dimension) of these two variables must be
            # constant for all questions (first dimension), and in the previous few lines we ensured it was
            # always 64. 
            # 
            # For each token in a question, the final hidden layer of the default RoBERTa model has 768 nodes.
            # The first line below returns a tensor with dimensions (batch_size, sequence_length=64, hidden_size=768),
            # where for each question in the batch, and each integer input token id out of the 64 tokens in the
            # question, the real values at the 768 hidden nodes are given. These 768 real values are the embedding for
            # one token from one question. We then average the embeddings for all non-padded tokens (i.e. where the attention
            # mask is 1), for every question in the batch. This gives a (batch_size, 768) dimensional question_embeddings
            # variable with the 768-dimensional embedding for all questions in the batch.
            with torch.no_grad():
                roberta_last_hidden_states = roberta_model(
                    input_ids,
                    attention_mask=attention_mask
                ).last_hidden_state
                mask = attention_mask.unsqueeze(-1).float()
                question_embeddings = torch.sum(roberta_last_hidden_states * mask, dim=1) / torch.clamp(mask.sum(dim=1), min=1e-9)

            roberta_question_embeddings[start:start + len(batch_questions)] = (
                question_embeddings.cpu()
            )

            if verbose:
                batch_number = start // batch_size + 1
                print(
                    f"RoBERTa batch {batch_number}/{total_batches} | "
                    f"Processed {start + len(batch_questions)}/{len(questions)} questions",
                    flush=True
                )

        return roberta_question_embeddings, question_to_idx

##### **Answer Selection Module**

This module is less involved than the previous two. For an unseen question (where we can’t search the training set for the correct answer), we first embed the question with the learnt question embedder. Then, for each answer entity in the KG, we find the value when the question triple (topic entity, question, answer entity) is represented as a triple of the corresponding, ordered embeddings and passed into the scoring metric. Note that the topic entity (and therefore its embedding) is obtained from the question and is known, and so is the question and its embedding – the only variable is the answer entity. The final answer is the answer entity that has the highest output value from the scoring metric. If required, a SoftMax can be performed over all raw outputs from the scoring metric (one output for each answer entity) to calculate probabilities for all answers.

In [ ]:
import torch
from collections.abc import Iterable

class answer_selection_module():
    """Answer selection module, which uses learnt embeddings from the KG embedding module
    and the learnt question embedding process from the question embedding module.
    
    Answer selection involves the ComplEx scoring function, where for a fixed topic entity
    string h and question string q, we pass the question triple (h, q, a) into the
    question_embedding_module.forward(), for all possible answer entities a in the knowledge graph.
    (((We use predict_proba() to give us a number between 0 to 1 for each question triple, that
    we can interpret as the triple's probability of correctness, but this is equivalent to
    question_embedding_module.forward() because the sigmoid function is always increasing))). Finally,
    the most likely answer entity is returned.
    """

    def __init__(self, kg_embedding_module: kg_embedding_module, question_embedding_module: question_embedding_module):
        """Initialise the ComplEx embedding model.
        
        Args:
            kg_embedding_module (kg_embedding_module class): the trained embedding model for entities and
                relations, containing the learned embeddings for all entities and relations in the knowledge graph
            question_embedding_module (question_embedding_module class): the trained question embedding module
                containing the learnt 4-layer neural network to convert question strings into complex vector embeddings
        
        Returns:
            None: Initialises the learnt embedding classes/models
        """

        self.kg_embedding_module = kg_embedding_module
        self.question_embedding_module = question_embedding_module

    def select_answer(self, batch: Iterable[tuple[str, str]]) -> list[str]:
        """Selects the answer for each question in the batch Iterable, using the method described
        at the top of this class

        Args:
            batch (Iterable[tuple[str, str]]): A collection of
                (topic entity, question) tuples represented as strings.
                Accepts a list, tuple, or set of tuples. The size of this iterable
                is batch_size.
                
        Returns:
            list[str]: A list containing the predicted answer entity for each question. The length
                of the returned list is batch_size.
                
        """
        entities = list(self.kg_embedding_module.get_all_entities())
        batch = list(batch)

        # Generate candidates for every question
        candidates = [
            (topic, question, entity)
            for topic, question in batch
            for entity in entities
        ]

        # Predict scores in one batch
        probabilities = self.question_embedding_module.forward(
            candidates
        )

        # Reshape scores into (number of questions, number of entities)
        probabilities = probabilities.reshape(len(batch), len(entities))

        # Find the highest-scoring entity for each question
        best_indices = torch.argmax(probabilities, dim=1).tolist()

        return [entities[index] for index in best_indices]

    def select_answer_fast(self, batch: Iterable[tuple[str, str]]) -> list[str]:
        """Selects the answer for each question in the batch Iterable, using the method described
        at the top of this class. Does it faster by accessing the pre-computed RoBERTa embeddings
        for questions seen in the training, validation and test data. This method does not work
        if a question in this batch is not in self.question_embedding_module.question_to_idx.keys().
        
        The fast answer selection provided here is another contribution we have made to
        the original Paper 2 model. 

        Args:
            batch (Iterable[tuple[str, str]]): A collection of
                (topic entity, question) tuples represented as strings.
                Accepts a list, tuple, or set of tuples. The size of this iterable
                is batch_size.
                        
        Returns:
            list[str]: A list containing the predicted answer entity for each question. The length
                of the returned list is batch_size.
        """

        batch = list(batch)

        if not batch:
            return []

        qem = self.question_embedding_module
        kg = self.kg_embedding_module

        qem.eval()

        device = next(qem.parameters()).device

        topics, questions = zip(*batch)

        # Ensure all questions have precomputed RoBERTa embeddings
        assert qem.question_to_idx is not None
        assert qem.roberta_question_embeddings is not None
        missing = set(questions) - qem.question_to_idx.keys()

        if missing:
            raise ValueError(
                f"{len(missing)} questions missing from RoBERTa cache"
            )

        # Retrieve each question embedding only once
        question_indices = torch.tensor(
            [qem.question_to_idx[q] for q in questions],
            dtype=torch.long,
            device=qem.roberta_question_embeddings.device
        )

        with torch.inference_mode():

            roberta_embeddings = (
                qem.roberta_question_embeddings[question_indices].to(device)
            )

            # Generate complex question embeddings
            r = qem.linear_relu_stack(roberta_embeddings)

            # Retrieve topic entity embeddings
            topic_indices = torch.tensor(
                [kg.entity_to_idx[t] for t in topics],
                dtype=torch.long,
                device=device
            )

            h = kg.entity_embeddings(topic_indices)

            # Retrieve ALL candidate entity embeddings
            all_entities = kg.entity_embeddings.weight

            # Split into real and imaginary components
            h_re, h_im = torch.chunk(h, 2, dim=-1)
            r_re, r_im = torch.chunk(r, 2, dim=-1)

            t_re, t_im = torch.chunk(all_entities, 2, dim=-1)

            # ComplEx scoring
            re_score = h_re * r_re - h_im * r_im
            im_score = h_re * r_im + h_im * r_re

            # Score every candidate entity using matrix multiplication
            scores = (
                re_score @ t_re.T
                + im_score @ t_im.T
            )

            # Select the highest-scoring entity
            best_indices = torch.argmax(scores, dim=1).tolist()

        # Maintain correspondence between entity indices and entity names
        entities = list(kg.entity_to_idx.keys())

        return [entities[index] for index in best_indices]
            


## Results:
This one is super easy to drop marks and its not actually that much work. Just show data of shit, good and tweaked models. I assume data will just be hits@1 for 1, 2 and 3 hop. Try to use alot of graphs and tables to show how data improved. At this stage we can lightly explain the data and make some slight comaprisons to all the models we've done. The next section is discussion so I will see how much we actually need to do. Results and special formatting for jupyter can be very easily ai'ed to make it look alot better.

### Paper 2:

A few things to note before we discuss Paper 2 results. As we have previously mentioned to our tutor, the computational cost of training and evaluating our Paper 2 model is very high, largely due to the significant size of the data (on the order of hundreds of thousands) and the high-dimensional embeddings for each entity, relation and question. Thus, it is not feasible to include executed "results" code cells in our notebook for the model trained on the entire training set and evaluated on the entire validation set, for 2-hop and 3-hop data. We have tried to use cloud computing (within a budget) and free Kaggle GPUs before they timeout, but have still not managed to obtain numerical results for some cases. The solution in this notebook is to train the 1-hop model with fewer epochs and training data, but this compromises on the final accuracy, which may not match the accuracy displayed on the presentation slides for 1-hop Paper 2.

The code for 1-hop training and validation set evaluation is given below.



In [ ]:
def load_qa_paper_2(root: Path, split: str) -> list[Example]:
    path = root / f"qa_{split}.txt"
    if not path.is_file():
        raise FileNotFoundError(f"QA file could not be found at path={path}")
    result: list[Example] = []
    with path.open(encoding="utf-8-sig") as file:
        for line_number, data in enumerate(file, 1):
            data = data.strip()
            if not data:
                continue
            try:
                question, answers = data.split("\t", maxsplit=1)
            except ValueError as e:
                raise ValueError(r"Expected QUESTION\tANSWERS at" + f"{path}:{line_number}") from e
            result.append(parse_question(question, answers))
    return result

In [ ]:
import sys
import argparse
import random
import math
from pathlib import Path

PROJECT_ROOT = Path("/kaggle/working")
MODELS_DIR = PROJECT_ROOT / "models"
sys.path.insert(0, str(MODELS_DIR))

"""
Description:
    Trains a knowledge graph question-answering model using random-search
    hyperparameter tuning.

    Evaluates different combinations of:
        - KG entity/relation and question complex embedding dimensions
        - Learning rates for both the KG and question embedding modules
        - Batch sizes
        - Number of train and validation questions
        - Negative samples per positive KG triple in the KG embedding module
        - Question encoder hidden layer dimensions
        - Question encoder weight decay
        - Question encoder node dropout

    Selects and prints the best hyperparameters based on validation accuracy.
"""
# Setup Paper 2 model training and validation
# parser = argparse.ArgumentParser()
# parser.add_argument("--root", type=Path, default=PROJECT_ROOT / "data")
# parser.add_argument("--hops", type=int, default=1)
# parser.add_argument("--kgem_epochs", type=int, default=100)
# parser.add_argument("--qem_epochs", type=int, default=100)
# parser.add_argument("--train", type=int, default = None)
# parser.add_argument("--val", type=int, default = None)
# parser.add_argument("--trials", type=int, default=30,
#                 help="Number of random hyperparameter configurations to evaluate")
# parser.add_argument("--no_cuda", action="store_true", default=False)
# args = parser.parse_args()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

DATA_ROOT = Path("/kaggle/input/datasets/aditmahi")
root = DATA_ROOT / "1-hop-vanilla"
hops = 1
kgem_epochs = 50
qem_epochs = 50
train = None
val = None
trials = 10

random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

kb_path = DATA_ROOT / "knowledge-graph" / "kb.txt"
if not kb_path.is_file():
    raise FileNotFoundError(f"kb file not found at path: kb_path={kb_path}")
graph = KnowledgeGraph(kb_path)
training = load_qa(root, hops, "train")
validation = load_qa(root, hops, "dev")

if train is not None and train <= len(training):
    training = random.sample(training, train)
else:
    training = random.sample(training, len(training))
if val is not None and val <= len(validation):
    validation = random.sample(validation, val)
else:
    validation = random.sample(validation, len(validation))

cache_path = Path("/kaggle/input/datasets/aditmahi/roberta-cache") / f"{hops}-hop.pt"
cache = torch.load(
    cache_path,
    map_location="cpu",
    weights_only=True
)
roberta_question_embeddings = cache["roberta_question_embeddings"]
question_to_idx = cache["question_to_idx"]

# Hyperparameter search space
embedding_dims = (20, 200)
batch_sizes = (32, 256)
kgem_learning_rates = (1e-6, 1e-2)
kgem_negative_samples = (1, 15)
qem_learning_rates = (1e-6, 1e-2)
qem_negative_samples = (1, 15)
hidden_dims = [
    (1024, 768, 512),
    (768, 512, 256),
    (768, 550, 400),
    (768, 500, 300),
    (600, 400, 200),
    (512, 256, 128),
    (300, 200, 100)
]
dropout_rates = (0.0, 0.5)
weight_decays = (1e-8, 1e-3)

best_accuracy = 0
best_params = None

for trial in range(trials):
    # Sample hyperparameters from the search space
    embedding_dim = random.randint(*embedding_dims)
    batch_size = random.randint(*batch_sizes)
    kgem_learning_rate = 10 ** random.uniform(
        math.log10(kgem_learning_rates[0]),
        math.log10(kgem_learning_rates[1])
    )
    kgem_negatives = random.randint(*kgem_negative_samples)
    qem_learning_rate = 10 ** random.uniform(
        math.log10(qem_learning_rates[0]),
        math.log10(qem_learning_rates[1])
    )
    qem_negatives = random.randint(*qem_negative_samples)
    hidden = random.choice(hidden_dims)
    dropout_rate = random.uniform(*dropout_rates)
    weight_decay = 10 ** random.uniform(
        math.log10(weight_decays[0]),
        math.log10(weight_decays[1])
    )

    print(f"\nTrial {trial + 1}/{trials}")
    print(
        f"Embedding dimension: {embedding_dim}, "
        f"Batch size: {batch_size}, "
        f"KG learning rate: {kgem_learning_rate:.6g}, "
        f"Negatives per positive KG triple: {kgem_negatives}, "
        f"Question learning rate: {qem_learning_rate:.6g}, "
        f"Negatives per positive question triple: {qem_negatives}, "
        f"Hidden layer dimensions in question embedding module: {hidden}, "
        f"Dropout in question embedding module: {dropout_rate:.3f}, "
        f"Weight decay in question embedding module: {weight_decay:.6g}"
    )

    # Train KG embedding module
    kg_model = kg_embedding_module(
        kg=graph,
        embedding_dim=embedding_dim
    ).to(device)

    kg_model.fit(
        epochs=kgem_epochs,
        batch_size=batch_size,
        learning_rate=kgem_learning_rate,
        negatives_per_positive=kgem_negatives,
    )

    # Train question embedding module
    question_model = question_embedding_module(
        kg_embedding_module=kg_model,
        question_training_data=training,
        question_to_idx=question_to_idx,
        roberta_question_embeddings=roberta_question_embeddings,
        first_hidden_layer_dim=hidden[0],
        second_hidden_layer_dim=hidden[1],
        third_hidden_layer_dim=hidden[2],
        dropout_rate=dropout_rate,
    ).to(device)

    question_model.fit(
        epochs=qem_epochs,
        batch_size=batch_size,
        learning_rate=qem_learning_rate,
        weight_decay=weight_decay,
        negatives_per_positive=qem_negatives
    )

    # Validation, using the answer selection module
    answer_selector = answer_selection_module(
        kg_model,
        question_model
    )

    correct = 0
    total = len(validation)
    # The sampled batch_size is not important for non-training purposes
    print("Beginning validation set performance testing")
    for start in range(0, total, 32):
        batch_correct = 0
        batch = validation[start:start + 32]

        predictions = answer_selector.select_answer_fast(
            [(example.topic, example.question) for example in batch]
        )

        for prediction, example in zip(predictions, batch):
            example_correct = int(prediction in example.answers)
            batch_correct += example_correct
            correct += example_correct

        processed = min(start + 32, total)

        print(
            f"Progress: {processed}/{total} "
            f"({processed / total * 100:.1f}%) | "
            f"Total Accuracy: {correct / processed:.4f} | "
            f"Batch Accuracy: {batch_correct / len(batch)}"
        )

    accuracy = correct / total

    print(f"\nFinal validation accuracy: {accuracy:.4f}")

    if accuracy > best_accuracy:
        best_accuracy = accuracy

        best_params = {
                "embedding_dimension": embedding_dim,
                "kg embedding module learning_rate": kgem_learning_rate,
                "question embedding module learning_rate": qem_learning_rate,
                "batch_size": batch_size,
                "negatives per positive KG triple": kgem_negatives,
                "question embedding module hidden_dimensions": hidden,
                "dropout_rate": dropout_rate,
                "weight_decay": weight_decay,
                "negatives per positive question triple": qem_negatives 
        }

print("\nBest hyperparameters:")
print(best_params)
print(f"Best validation accuracy: {best_accuracy:.4f}")

## Discussion
As I said, this one is pretty easy. Talk about your results, look at papers, really look at limitations and why they apear. Where possible offer alteratives to fix the probelsms or talk about differnt models fixing the problem but their other downsides i.e I am pretty sure one of the papers is more effective at 3 hop but it kind doesn't do well a sparce graph. 
